# Products.csv — Data Exploration

Exploration of `products.csv` following the same structural → null/missingness → duplicate →
value-range checklist used for `reviews.csv`. In progress — covers structural profile, nulls,
`asin` uniqueness, and `availability`, `best_sellers_rank`, `brand_name`, `brand_page_url`,
`breadcrumbs`, and `default_variant/0-2`. Remaining columns and referential integrity against
`reviews.csv` to be completed in a follow-up session.

## Setup

In [178]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [179]:
data_path = Path("../data/raw")
products = pd.read_csv(data_path / "products.csv")
products.shape

(728, 34)

## 1. Structural Profile

In [180]:
products.head(1)

,s.no,about_item,asin,availability,best_sellers_rank,brand_name,brand_page_url,breadcrumbs,customer_review_summary,default_variant/0,default_variant/1,default_variant/2,delivery_date,fastest_delivery_date,list_price,manufacturer,model_number,price_value,product_description,product_url,rating_count,rating_distribution/1star,rating_distribution/2star,rating_distribution/3star,rating_distribution/4star,rating_distribution/5star,rating_stars,recent_purchases,scrape_time,seller_name,seller_page_url,title,all_images,rank_1
0,0,Premium Comfort: Crafted from a high-quality c...,B0B59BJG6Y,In Stock,"#56,836 in Clothing, Shoes & Jewelry (See Top ...",MLYENX Store,https://www.amazon.com/stores/MLYENX/page/1FCD...,"Clothing, Shoes & Jewelry › Men › Clothing › A...",Customers find the shirts comfortable and well...,size:Large,"color:5 Pack Black, Dark Grey, Light Blue, Mil...",NaN,"Saturday, March 15","Tomorrow, March 11",List Price: $53.99,NaN,NaN,39.9926,NaN,https://www.amazon.com/dp/B0B59BJG6Y,"1,654 ratings",2%,1%,7%,15%,75%,4.6 out of 5 stars,50+ bought,03-10-2025 21:42,Greenfive,https://www.amazon.com/gp/help/seller/at-a-gla...,4/5 Pack Mens Polo Shirts Short Sleeve Quick D...,['https://m.media-amazon.com/images/I/41yUF65P...,85.0


In [181]:
products_cols = list(products.columns)

## 2. Null / Missingness Pass

In [182]:
for col in products_cols:
    null_pct = round((products[col].isna().sum() / products.shape[0]) * 100, 2)
    print(f"Nulls percent in {col} is: {null_pct}")

Nulls percent in s.no is: 0.0
Nulls percent in about_item is: 0.0
Nulls percent in asin is: 0.0
Nulls percent in availability is: 1.79
Nulls percent in best_sellers_rank is: 23.35
Nulls percent in brand_name is: 0.0
Nulls percent in brand_page_url is: 10.85
Nulls percent in breadcrumbs is: 1.79
Nulls percent in customer_review_summary is: 12.91
Nulls percent in default_variant/0 is: 4.4
Nulls percent in default_variant/1 is: 9.62
Nulls percent in default_variant/2 is: 99.86
Nulls percent in delivery_date is: 3.57
Nulls percent in fastest_delivery_date is: 8.38
Nulls percent in list_price is: 44.92
Nulls percent in manufacturer is: 63.46
Nulls percent in model_number is: 72.66
Nulls percent in price_value is: 2.88
Nulls percent in product_description is: 62.77
Nulls percent in product_url is: 0.0
Nulls percent in rating_count is: 1.24
Nulls percent in rating_distribution/1star is: 0.0
Nulls percent in rating_distribution/2star is: 0.0
Nulls percent in rating_distribution/3star is: 0.0
N

## 3. Duplicate Pass

In [183]:
products["asin"].duplicated().sum()

np.int64(0)

`asin` is fine and there is no duplication inside it, which is a good sign.

## 4. Column-Specific Notes

### `availability`

In [184]:
products["availability"].value_counts()

availability
In Stock                                       640
Currently unavailable.                          18
In stock                                        16
Only 1 left in stock - order soon.              14
Only 3 left in stock - order soon.              10
Only 5 left in stock - order soon.               6
Only 2 left in stock - order soon.               6
Available to ship in 1-2 days                    2
This item will be released on May 20, 2025.      1
Only 4 left in stock - order soon.               1
Temporarily out of stock.                        1
Name: count, dtype: int64

`availability` has several issues that need unifying:

1. `In Stock` and `In stock` are the same value with a capital letter making the difference
   (noted for the transformation step).
2. Almost all values will need to collapse to one of 2 options — in stock or unavailable —
   or roughly 3 options: in stock, unavailable, and limited.

### `best_sellers_rank`

In [185]:
for rank in products["best_sellers_rank"].dropna().sample(3).values:
    print(rank.split())

['#473', 'in', 'Our', 'Brands', '(See', 'Top', '100', 'in', 'Our', 'Brands)', '#1', 'in', "Men's", 'Down', 'Jackets', '&', 'Coats']
['#5,623', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry', '(See', 'Top', '100', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry)', '#19', 'in', "Men's", 'Polo', 'Shirts', '#245', 'in', "Men's", 'Golf', 'Shirts']
['#3,789', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry', '(See', 'Top', '100', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry)', '#49', 'in', "Women's", 'Tunics']


`best_sellers_rank` contains the rank of the product in the best sellers list, and what
matters is the numbers (the ranks). For the transformation step:

1. Take the numbers and provide a new column for the category.
2. Handle the number string → int conversion cleanly.

### `brand_name`

In [186]:
products["brand_name"].sample(5)

116    Wrangler Authentics Store
129        Rock & Republic Store
442            Brand: Over Crowd
643                 adidas Store
337                  Brand: Nike
Name: brand_name, dtype: str

In [187]:
mask = ~products["brand_name"].str.contains("store|brand", case=False, na=True)

single_name = products.loc[mask, "brand_name"].iloc[0]
print(single_name)

Unknown


`brand_name` notes:

1. 79 brand names contain "Store" in them, which will need removing with the pattern
   `"xxx Store"`.
2. The rest contain the word "Brand" at the start, with the pattern `"Brand: xxx"`.
3. One name is literally the string `"Unknown"`, which should be treated as NaN/null —
   to handle in the Transformer.

### `brand_page_url`

In [188]:
urls = products["brand_page_url"]

# Filter for common URL structural anomalies
anomalies = products[
    urls.isna() |
    ~urls.str.startswith(("http://", "https://"), na=False) |
    urls.str.contains(r"\s", na=False) |
    urls.str.contains(r"[<>{}\\|\\^~\[\]`]", na=False)
]

print(f"Found {len(anomalies)} potential anomalies:")
print(anomalies[["brand_name", "brand_page_url"]])

Found 79 potential anomalies:
                 brand_name brand_page_url
22   Brand: U.S. Polo Assn.            NaN
27   Brand: U.S. Polo Assn.            NaN
29   Brand: U.S. Polo Assn.            NaN
31   Brand: U.S. Polo Assn.            NaN
49          Brand: KAOKLRNI            NaN
..                      ...            ...
696           Brand: Gerber            NaN
697           Brand: Gerber            NaN
704          Brand: Generic            NaN
712          Brand: KONQUWA            NaN
723           Brand: QICAMO            NaN

[79 rows x 2 columns]


No suspicious links found, only some nulls — safe to go with it as-is.

### `breadcrumbs`

In [189]:
products["breadcrumbs"]

0      Clothing, Shoes & Jewelry › Men › Clothing › A...
1      Clothing, Shoes & Jewelry › Men › Clothing › S...
2      Clothing, Shoes & Jewelry › Men › Clothing › A...
3      Clothing, Shoes & Jewelry › Men › Clothing › A...
4      Clothing, Shoes & Jewelry › Men › Clothing › A...
                             ...                        
723    Clothing, Shoes & Jewelry › Women › Clothing ›...
724    Clothing, Shoes & Jewelry › Sport Specific Clo...
725    Clothing, Shoes & Jewelry › Women › Clothing ›...
726    Clothing, Shoes & Jewelry › Women › Clothing ›...
727    Clothing, Shoes & Jewelry › Women › Clothing ›...
Name: breadcrumbs, Length: 728, dtype: str

In [190]:
products["breadcrumbs"].loc[
    products["breadcrumbs"].isna() & products["best_sellers_rank"].notna()
]

Series([], Name: breadcrumbs, dtype: str)

Notes:

1. No need to work much on the `best_sellers_rank` column since `breadcrumbs` already gives
   the category with a suitable separator (the `›` trail) to go with.
2. Worth noting: when `breadcrumbs` is null, `best_sellers_rank` is also null — but not the
   other way around, which rules out using `breadcrumbs` to compensate for the missing
   `best_sellers_rank` values.

### `default_variant/0`, `/1`, `/2`

In [191]:
for col in products_cols:
    if "default_variant" in col:
        print(products[col].sample(5))
        print("=" * 40)

211              size:34
707         size:14 Long
209              size:34
629                  NaN
67     size:4X-Large Big
Name: default_variant/0, dtype: str
400                                     color:White/Navy
0      color:5 Pack Black, Dark Grey, Light Blue, Mil...
579                                          color:Khaki
475                                          color:Beige
689                                       color:A- Green
Name: default_variant/1, dtype: str
592    NaN
110    NaN
218    NaN
136    NaN
273    NaN
Name: default_variant/2, dtype: str


In [192]:
(~products["default_variant/0"].str.contains("size", case=False, na=True)).sum()

np.int64(11)

In [194]:
# 1. Create the boolean mask for rows that don't contain "size" and aren't NaN
mask = ~products["default_variant/0"].str.contains("size", case=False, na=True)

# 2. Print just the unique variant values to see what they are
print("Values:")
print(products.loc[mask, "default_variant/0"].unique())

# 3. View the full DataFrame for these rows
no_size_df = products[mask]

Values:
<StringArray>
['style:6" Boxer Brief Fly Front With Pouch',                               'color:Black',                         'color:Neon Orange',                               'color:Green',
                             'color:Magenta',                   'color:Jet Black & Beige',                              'color:Maroon',                          'color:Rama Green',
                             'color:Mustard',                          'color:Light Pink']
Length: 10, dtype: str


Most of `default_variant/0` is the size of the variant, with a small amount of pollution
containing the "color" instead — same situation for `default_variant/1`, which also
contains some pollution.

## Next Steps

Remaining columns not yet profiled: `customer_review_summary`, `delivery_date`,
`fastest_delivery_date`, `list_price`, `manufacturer`, `model_number`, `price_value`,
`product_description`, `product_url`, `rating_count`, `rating_distribution/1star`–`5star`,
`rating_stars`, `recent_purchases`, `scrape_time`, `seller_name`, `seller_page_url`, `title`,
`all_images`, `rank_1`.

Referential integrity against `reviews.csv` (orphan `asin` values, reviews-per-product
cardinality) deferred to a joint pass once both files are fully profiled.